### 1:

In [2]:
import sqlite3 
import pandas as pd
db_ref = 'AdventureWorks.db'

In [3]:


queryQ1 = '''

SELECT MIN(p.ListPrice) AS min_price, MAX(p.ListPrice) AS max_price, MAX(p.ListPrice) - MIN(p.ListPrice) as price_diff, pc.ParentProductCategoryID, pc.Name, COUNT(*) AS 'type quantity'
FROM productcategory as pc
JOIN product as p
ON p.ProductCategoryID=pc.ProductCategoryID
GROUP BY pc.ParentProductCategoryID
'''

with sqlite3.connect(db_ref) as conn:
        price_ranges = pd.read_sql_query(queryQ1, conn)

price_ranges.head()

,min_price,max_price,price_diff,ParentProductCategoryID,Name,type quantity
0,539.99,3578.27,3038.28,1,Road Bikes,97
1,20.24,1431.50,1411.26,2,Road Frames,134
2,8.99,89.99,81.00,3,Bib-Shorts,35
3,2.29,159.00,156.71,4,Bike Stands,29


In [4]:
sql_query = '''
    SELECT ParentProductCategoryID,
        (SELECT name
        FROM ProductCategory
        WHERE (ProductCategoryID == pc.ParentProductCategoryID)
        ) as Name, MIN(p.ListPrice) AS min_price, MAX(p.ListPrice) AS max_price, MAX(p.ListPrice) - MIN(p.ListPrice) as price_diff, COUNT(*) AS 'type quantity'
    FROM ProductCategory AS pc
        JOIN Product AS p
        ON pc.ProductCategoryID = p.ProductCategoryID
    GROUP BY pc.ParentProductCategoryID
            '''

with sqlite3.connect(db_ref) as conn:
    df = pd.read_sql_query(sql_query, conn)
df.head()

,ParentProductCategoryID,Name,min_price,max_price,price_diff,type quantity
0,1,Bikes,539.99,3578.27,3038.28,97
1,2,Components,20.24,1431.50,1411.26,134
2,3,Clothing,8.99,89.99,81.00,35
3,4,Accessories,2.29,159.00,156.71,29


### 2. Price Ranges by Subcategory

In [5]:
from sympy import product

queryQ1 = '''

SELECT MIN(p.ListPrice) AS min_price, MAX(p.ListPrice) AS max_price, MAX(p.ListPrice) - MIN(p.ListPrice) as price_diff, pc.name, COUNT(*) AS 'type quantity'
FROM product as p
JOIN productcategory as pc
ON p.ProductCategoryID=pc.ProductCategoryID
GROUP BY p.ProductCategoryID
'''

with sqlite3.connect(db_ref) as conn:
        price_ranges = pd.read_sql_query(queryQ1, conn)

price_ranges.head()

,min_price,max_price,price_diff,Name,type quantity
0,539.99,3399.99,2860.00,Mountain Bikes,32
1,539.99,3578.27,3038.28,Road Bikes,43
2,742.35,2384.07,1641.72,Touring Bikes,22
3,44.54,120.27,75.73,Handlebars,8
4,53.99,121.49,67.50,Bottom Brackets,3


In [6]:
queryQ2 = '''
SELECT *
FROM product
'''
with sqlite3.connect(db_ref) as conn:
    price_basket = pd.read_sql_query(queryQ2, conn)


price_basket["Price Bracket"] = pd.cut(
    price_basket["ListPrice"],
    bins=[0, 50, 100, 500, 1000, float("inf")],
    labels=["Under $50","Under $100", "Under $500", "Under $1000", "Over $1000"],
    right=False
)

price_basket.sample(10)

,ProductID,Name,ProductNumber,Color,StandardCost,ListPrice,Size,Weight,ProductCategoryID,ProductModelID,SellStartDate,SellEndDate,DiscontinuedDate,ThumbNailPhoto,ThumbnailPhotoFileName,rowguid,ModifiedDate,Price Bracket
61,766,"Road-650 Black, 60",BK-R50B-60,Black,486.7066,782.99,60,9026.44,6,30,2005-07-01 00:00:00.000,2007-06-30 00:00:00.000,,47494638396150003100F70000EEEDF8F2F1F2EDEDF5E0...,superlight_black_small.gif,A2DB196D-6640-49EA-A84F-2E87CA6F50C6,2008-03-11 10:01:36.827,Under $1000
115,820,HL Road Front Wheel,FW-R820,Black,146.5466,330.06,,650,21,51,2006-07-01 00:00:00.000,2007-06-30 00:00:00.000,,47494638396133003200F700009D9E9EFEFCFCF23735FE...,wheel_small.gif,727A3CD5-8D40-4884-A7E4-DFD3FFDEB164,2008-03-11 10:01:36.827,Under $500
154,859,"Half-Finger Gloves, M",GL-H102-M,Black,9.1593,24.49,M,,24,4,2006-07-01 00:00:00.000,,,47494638396150003100F7000000000080000000800080...,no_image_available_small.gif,9D458FD5-392D-4AB1-AFEF-6A5548E48858,2008-03-11 10:01:36.827,Under $50
182,887,"HL Touring Frame - Yellow, 46",FR-T98Y-46,Yellow,601.7437,1003.91,46,1342.63,20,7,2007-07-01 00:00:00.000,,,47494638396150003100F7000000000080000000800080...,no_image_available_small.gif,137675A7-34CD-4B7A-ABE1-4E0EEB79B65D,2008-03-11 10:01:36.827,Over $1000
23,728,"LL Road Frame - Red, 58",FR-R38R-58,Red,187.1571,337.22,58,1115.83,18,9,2005-07-01 00:00:00.000,2007-06-30 00:00:00.000,,47494638396150003100F7000000000080000000800080...,no_image_available_small.gif,799A56FF-5AD2-41B3-BFAC-528B477AD129,2008-03-11 10:01:36.827,Under $500
225,930,HL Mountain Tire,TI-M823,,13.0900,35.00,,,41,87,2007-07-01 00:00:00.000,,,47494638396150003200F700007AA31DCEA945F9F6EEBA...,mb_tires_small.gif,DDAD25F5-0445-4B5C-8466-6446930AD8B8,2008-03-11 10:01:36.827,Under $50
156,861,"Full-Finger Gloves, S",GL-F110-S,Black,15.6709,37.99,S,,24,3,2006-07-01 00:00:00.000,2007-06-30 00:00:00.000,,47494638396150003100F7000000000080000000800080...,no_image_available_small.gif,76FAC097-1FB3-456B-8FB9-2C7A613771B4,2008-03-11 10:01:36.827,Under $50
172,877,Bike Wash - Dissolver,CL-9009,,2.9733,7.95,,,33,119,2007-07-01 00:00:00.000,,,47494638396150003100F7000000000080000000800080...,no_image_available_small.gif,3C40B5AD-E328-4715-88A7-EC3220F02ACF,2008-03-11 10:01:36.827,Under $50
228,933,HL Road Tire,TI-R982,,12.1924,32.60,,,41,90,2007-07-01 00:00:00.000,,,47494638396150003200F70000FCFCFC2F02023E0202BE...,street_tires_small.gif,C86DE56A-5048-4B89-B7C7-56BC75C9BCEE,2008-03-11 10:01:36.827,Under $50
125,830,"ML Mountain Frame - Black, 40",FR-M63B-40,Black,185.8193,348.76,40,1256.44,16,14,2006-07-01 00:00:00.000,2007-06-30 00:00:00.000,,47494638396150003100F7000000000080000000800080...,no_image_available_small.gif,AED1C9C4-C139-45D2-B38E-0A0A9F518665,2008-03-11 10:01:36.827,Under $500


In [7]:
price_basket['Price Bracket'].value_counts()

Price Bracket
Over $1000     86
Under $500     69
Under $50      53
Under $1000    50
Under $100     37
Name: count, dtype: int64

In [8]:
queryQ2 = '''
SELECT
CASE
WHEN ListPrice < 100 THEN 'Cheap'
WHEN ListPrice BETWEEN 1000 AND 2000 THEN 'Medium'
ELSE 'Expensive'
END AS Price_Basket,
COUNT(*) AS Product_Count
FROM product
GROUP BY Price_Basket;
'''
with sqlite3.connect(db_ref) as conn:
    price_basket = pd.read_sql_query(queryQ2, conn)

price_basket.head()



,Price_Basket,Product_Count
0,Cheap,90
1,Expensive,154
2,Medium,51


## HR Questions!!!

### Question 1

In [9]:
db_employees = 'employees.db'


In [10]:
Query3 = '''
SELECT *
FROM employees
WHERE vacation_hours > 40
'''

with sqlite3.connect(db_employees) as conn:
        employee_hours = pd.read_sql_query(Query3, conn)

employee_hours.head()

,customer_id,email_address,vacation_hours,sallaried,title
0,2,keith0@adventure-works.com,80,1,manager
1,3,donna0@adventure-works.com,69,1,sales rep
2,4,janet1@adventure-works.com,49,1,sales rep
3,5,lucy0@adventure-works.com,49,1,sales rep
4,6,rosmarie0@adventure-works.com,88,1,manager


In [11]:
employee_hours['firstname']= employee_hours['email_address'].str.split("@").str[0]
employee_hours['firstname']

0         keith0
1         donna0
2         janet1
3          lucy0
4      rosmarie0
         ...    
549    michael25
550    margaret2
551        gary6
552    patricia2
553        raja0
Name: firstname, Length: 554, dtype: object

In [12]:
employee_hours.head()

,customer_id,email_address,vacation_hours,sallaried,title,firstname
0,2,keith0@adventure-works.com,80,1,manager,keith0
1,3,donna0@adventure-works.com,69,1,sales rep,donna0
2,4,janet1@adventure-works.com,49,1,sales rep,janet1
3,5,lucy0@adventure-works.com,49,1,sales rep,lucy0
4,6,rosmarie0@adventure-works.com,88,1,manager,rosmarie0


In [13]:
employee_hours.to_excel('employee_hours.xlsx', index=False)

In [14]:
Query4 = '''
SELECT title, sallaried, vacation_hours, AVG(vacation_hours)
FROM employees
WHERE title LIKE '%sales%'
GROUP BY sallaried

'''

with sqlite3.connect(db_employees) as conn:
        salaried = pd.read_sql_query(Query4, conn)

salaried



,title,sallaried,vacation_hours,AVG(vacation_hours)
0,sales rep,0,18,55.600000
1,sales rep,1,69,53.133929


In [15]:
Query5 = '''
SELECT title, sallaried, AVG(vacation_hours)
FROM employees
WHERE title LIKE '%sales%'
GROUP BY sallaried
UNION ALL 
SELECT title, sallaried,AVG(vacation_hours)
FROM employees
WHERE title LIKE '%manager%'
GROUP BY sallaried

'''

with sqlite3.connect(db_employees) as conn:
        salaried = pd.read_sql_query(Query5, conn)

salaried


,title,sallaried,AVG(vacation_hours)
0,sales rep,0,55.600000
1,sales rep,1,53.133929
2,manager,0,58.715789
3,manager,1,54.687805


In [16]:
salaried.to_excel('salaries.xlsx', index=False)

## Marketing Team Questions

### Question 1

In [61]:


Query6 = '''
SELECT c.CustomerID, c.Title || ' ' || c.FirstName || ' ' || c.MiddleName || ' ' || c.LastName  AS Name, s.OrderDate
FROM customer AS c
RIGHT JOIN SalesOrderHeader AS s
        ON s.CustomerID = c.CustomerID
WHERE strftime('%m', OrderDate) = '06'
'''

with sqlite3.connect(db_ref) as conn: 
    conn.text_factory = lambda x: x.decode('latin-1', errors='replace')
    customers = pd.read_sql_query(Query6, conn)

customers.sample(10)
customers.to_excel('customer_sales.xlsx', index=False)